In [ ]:
import re, warnings
import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.model_selection import KFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import gc
import time

warnings.filterwarnings("ignore")

TRAIN_PATH = "train.csv"
TEST_PATH  = "test.csv"
SUB_PATH   = "submission.csv"
SEED       = 42
N_FOLDS    = 5
SMOOTH_K   = 15

total_start = time.time()

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

target_col = "salary_mean_net"
y_raw = train[target_col].values.astype(np.float64)
y = np.log1p(y_raw)
G = y.mean()

all_df = pd.concat([train.drop(columns=[target_col]), test], ignore_index=True)
n_tr = len(train)
n_te = len(test)

kf = KFold(N_FOLDS, shuffle=True, random_state=SEED)

def mape(yt, yp):
    return float(np.mean(np.abs((yt - yp) / np.maximum(yt, 1))))

def to_str(s):
    return "" if not isinstance(s, str) else s


def extract_nums(text):
    if not isinstance(text, str) or len(text) == 0:
        return []
    text = text.replace("\xa0", " ").lower()
    nums = set()
    for m in re.finditer(r"(\d{2,3})\s*(?:тыс|[kк])", text):
        v = int(m.group(1)) * 1000
        if 5000 <= v <= 1000000:
            nums.add(v)
    for m in re.finditer(r"\b(\d{5,6})\b", text):
        v = int(m.group(1))
        if 5000 <= v <= 1000000:
            nums.add(v)
    for m in re.finditer(r"от\s*(\d{2,3}\s?\d{3})", text):
        v = int(m.group(1).replace(" ", ""))
        if 5000 <= v <= 1000000:
            nums.add(v)
    for m in re.finditer(r"до\s*(\d{2,3}\s?\d{3})", text):
        v = int(m.group(1).replace(" ", ""))
        if 5000 <= v <= 1000000:
            nums.add(v)
    return sorted(nums) if nums else []

all_nums = all_df["raw_description"].fillna("").apply(extract_nums)

all_df["hint_min"]   = all_nums.apply(lambda x: float(min(x)) if x else np.nan)
all_df["hint_max"]   = all_nums.apply(lambda x: float(max(x)) if x else np.nan)
all_df["hint_med"]   = all_nums.apply(lambda x: float(np.median(x)) if x else np.nan)
all_df["hint_mean"]  = all_nums.apply(lambda x: float(np.mean(x)) if x else np.nan)
all_df["hint_count"] = all_nums.apply(len).astype(np.float32)
all_df["hint_has"]   = (~all_df["hint_med"].isna()).astype(np.int8)
all_df["hint_range"] = (all_df["hint_max"] - all_df["hint_min"]).fillna(0)
all_df["log_hint_min"] = np.log1p(all_df["hint_min"].fillna(0))
all_df["log_hint_med"] = np.log1p(all_df["hint_med"].fillna(0))
all_df["log_hint_max"] = np.log1p(all_df["hint_max"].fillna(0))

city_s = all_df["unified_address_city"].fillna("").str.lower()
region_s = all_df.get("unified_address_region", pd.Series("", index=all_df.index)).fillna("").str.lower()

all_df["is_moscow"] = (city_s.str.contains("москва", na=False) | region_s.str.contains("москва", na=False)).astype(np.int8)
all_df["is_spb"] = (city_s.str.contains("санкт", na=False) | region_s.str.contains("санкт-петербург", na=False)).astype(np.int8)
all_df["is_remote"] = (all_df.get("schedule_name", "") == "Удаленная работа").astype(np.int8)
all_df["has_skills"] = (all_df["key_skills_name"].fillna("не указано") != "не указано").astype(np.int8)
all_df["is_branded"] = (all_df.get("is_branded_description", "") == "заполнено").astype(np.int8)

it_kw = r"(разработ|программ|developer|devops|data|аналитик|финанс|банк|\bit\b|qa|тестир)"
all_df["is_it"] = all_df["name_clean"].fillna("").str.lower().str.contains(it_kw, na=False).astype(np.int8)

senior_kw = r"(старший|ведущий|главный|руководитель|директор|начальник|chief|senior|lead|head)"
all_df["is_senior"] = all_df["name_clean"].fillna("").str.lower().str.contains(senior_kw, na=False).astype(np.int8)

exp_map = {"Нет опыта": 0, "От 1 года до 3 лет": 1, "От 3 до 6 лет": 2, "Более 6 лет": 3}
all_df["exp_ord"] = all_df["experience_name"].map(exp_map).fillna(-1).astype(np.int8)

for col in ["professional_roles_name", "unified_address_city"]:
    if col in all_df.columns:
        freq_map = all_df[col].fillna("__NA__").value_counts()
        all_df[col + "_freq"] = all_df[col].fillna("__NA__").map(freq_map).fillna(0).astype(np.float32)


def smooth_encode(g_train, y_train, g_query, glob=G, k=SMOOTH_K):
    df = pd.DataFrame({"g": g_train, "y": y_train})
    stat = df.groupby("g")["y"].agg(["sum", "count"])
    stat["enc"] = (stat["sum"] + k * glob) / (stat["count"] + k)
    return np.array([stat.loc[v, "enc"] if v in stat.index else glob for v in g_query])

def fold_te_fast(series_all, y_tr, col_name):
    vals_tr = series_all.iloc[:n_tr].astype(str).values
    vals_te = series_all.iloc[n_tr:].astype(str).values
    oof = np.full(n_tr, G)
    for tr_i, va_i in kf.split(np.arange(n_tr)):
        oof[va_i] = smooth_encode(vals_tr[tr_i], y_tr[tr_i], vals_tr[va_i])
    all_df[col_name] = np.concatenate([oof, smooth_encode(vals_tr, y_tr, vals_te)])

for col in ["professional_roles_name", "unified_address_city", "experience_name",
            "specializations_profarea_name", "name_clean", "employer_industries"]:
    if col in all_df.columns:
        fold_te_fast(all_df[col], y, col + "_te")

if "employer_id" in all_df.columns:
    fold_te_fast(all_df["employer_id"].fillna(-1).astype(str), y, "employer_te")

combos = [
    ("professional_roles_name", "experience_name"),
    ("professional_roles_name", "unified_address_city"),
    ("name_clean", "unified_address_city"),
]

for ca, cb in combos:
    if ca in all_df.columns and cb in all_df.columns:
        combo = all_df[ca].fillna("_").astype(str) + "|||" + all_df[cb].fillna("_").astype(str)
        fold_te_fast(combo, y, f"{ca}_{cb}_te")


tfidf_name = TfidfVectorizer(max_features=15000, ngram_range=(1,2), min_df=3, sublinear_tf=True)
X_name = tfidf_name.fit_transform(all_df["name_clean"].apply(to_str))

desc_col = "lemmaized_wo_stopwords_raw_description" if "lemmaized_wo_stopwords_raw_description" in all_df.columns else "raw_description"
tfidf_desc = TfidfVectorizer(max_features=40000, ngram_range=(1,2), min_df=5, sublinear_tf=True)
X_desc = tfidf_desc.fit_transform(all_df[desc_col].apply(to_str))

tfidf_skills = TfidfVectorizer(max_features=5000, ngram_range=(1,1), min_df=3, sublinear_tf=True)
X_skills = tfidf_skills.fit_transform(all_df["key_skills_name"].apply(to_str))

svd_name = TruncatedSVD(100, random_state=SEED).fit(X_name)
svd_desc = TruncatedSVD(200, random_state=SEED).fit(X_desc)
svd_skills = TruncatedSVD(30, random_state=SEED).fit(X_skills)

X_name_d = svd_name.transform(X_name).astype(np.float32)
X_desc_d = svd_desc.transform(X_desc).astype(np.float32)
X_skills_d = svd_skills.transform(X_skills).astype(np.float32)


struct_cols = [
    "exp_ord", "is_moscow", "is_spb", "is_remote", "has_skills", "is_branded", 
    "is_it", "is_senior", "hint_min", "hint_max", "hint_med", "hint_mean",
    "hint_count", "hint_has", "hint_range", "log_hint_min", "log_hint_med", "log_hint_max",
    "professional_roles_name_freq", "unified_address_city_freq",
    "professional_roles_name_te", "unified_address_city_te", "experience_name_te",
    "specializations_profarea_name_te", "name_clean_te", "employer_industries_te",
]

if "employer_te" in all_df.columns:
    struct_cols.append("employer_te")

struct_cols += [f"{ca}_{cb}_te" for ca, cb in combos 
                if f"{ca}_{cb}_te" in all_df.columns]

struct_cols = [c for c in struct_cols if c in all_df.columns]

X_struct = all_df[struct_cols].fillna(0).astype(np.float32)
scaler = StandardScaler()
X_struct_scaled = scaler.fit_transform(X_struct.values).astype(np.float32)

X_sparse = hstack([csr_matrix(X_struct_scaled), X_name, X_desc, X_skills])

Xsp_tr = X_sparse[:n_tr]
Xsp_te = X_sparse[n_tr:]


del X_sparse, X_struct, X_struct_scaled
gc.collect()

print("Training LightGBM")

lgb1_oof = np.zeros(n_tr)
lgb1_test = np.zeros(n_te)

params1 = {
    'objective': 'regression_l1',
    'learning_rate': 0.05,
    'num_leaves': 127,
    'min_child_samples': 30,
    'feature_fraction': 0.7,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'n_jobs': -1,
    'random_state': SEED,
    'verbose': -1,
}

for fold, (tr, va) in enumerate(kf.split(Xsp_tr)):
    fold_start = time.time()
    print(f"  Fold {fold+1}/5...", end="", flush=True)
    
    dtrain = lgb.Dataset(Xsp_tr[tr], label=y[tr])
    dval = lgb.Dataset(Xsp_tr[va], label=y[va], reference=dtrain)
    
    model = lgb.train(params1, dtrain, num_boost_round=2000, valid_sets=[dval],
                      callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(False)])
    
    lgb1_oof[va] = model.predict(Xsp_tr[va])
    lgb1_test += model.predict(Xsp_te) / N_FOLDS
    
    fold_mape = mape(y_raw[va], np.expm1(lgb1_oof[va]))
    fold_time = (time.time() - fold_start) / 60
    print(f" trees={model.best_iteration:4d}, MAPE={fold_mape:.5f}, time={fold_time:.1f}min")

final_oof_mape = mape(y_raw, np.expm1(lgb1_oof))
print(f"\nOOF MAPE = {final_oof_mape:.5f}")

preds_test = np.expm1(lgb1_test)
preds_oof = np.expm1(lgb1_oof)

best_mape = mape(y_raw, preds_oof)
best_base = 100

for base in [50, 100, 500, 1000]:
    rounded = np.round(preds_oof / base) * base
    rounded = np.clip(rounded, 5000, 1000000)
    curr_mape = mape(y_raw, rounded)
    print(f"  Round {base}: MAPE={curr_mape:.5f}")
    if curr_mape < best_mape:
        best_mape = curr_mape
        best_base = base

print(f"Best round base: {best_base}")

preds_test = np.round(preds_test / best_base) * best_base
preds_test = np.clip(preds_test, 5000, 1000000)

sub = pd.DataFrame({
    "id": test["id"], 
    target_col: preds_test.astype(int)
})
sub.to_csv(SUB_PATH, index=False)

print(f"Saved: {SUB_PATH}")
print(f"Predictions: min={preds_test.min():.0f}, max={preds_test.max():.0f}, mean={preds_test.mean():.0f}")